#  Plant Disease Classifier
## Step 3: Transfer Learning & MobileNetV3

**What we'll do notebook:**
- What is Transfer Learning and why we use it
- What is MobileNetV3 and how it was trained
- How to load a pretrained model in PyTorch
- What freezing and unfreezing layers means
- Visualize the model architecture

---

---
##  Cell 1: What is Transfer Learning?

Imagine you already know how to drive a car.
Learning to drive a truck is now MUCH easier — you don't start from zero.

**Transfer Learning works the same way:**

```
WITHOUT Transfer Learning:
Random weights → Train on 20,000 images → Maybe 70% accuracy
Takes hours, needs millions of images

WITH Transfer Learning:
Pretrained weights (already knows edges, shapes, textures)
→ Fine-tune on 20,000 images → 95%+ accuracy
Takes minutes!
```

A pretrained model has already learned:
- Layer 1-3  : Basic edges, corners, colors
- Layer 4-7  : Shapes, textures, patterns
- Layer 8-12 : Complex features (eyes, leaves, wheels)
- Last layer : Final classification (cat, dog, car...)

We KEEP all the learned features and only REPLACE the last layer for our task!

In [1]:
# Let's visualize the concept of Transfer Learning
print(" Transfer Learning Concept")
print("=" * 50)
print()
print("MobileNetV3 was trained on ImageNet:")
print("    1.2 million images")
print("    1,000 different classes (cats, dogs, cars...)")
print("    Took days to train on powerful GPUs")
print()
print("We are going to:")
print("    KEEP all learned features (weights)")
print("    REPLACE only the last layer")
print("    TRAIN only the last layer on our leaf data")
print()
print("Result: Great accuracy with minimal training time! ")

 Transfer Learning Concept

MobileNetV3 was trained on ImageNet:
    1.2 million images
    1,000 different classes (cats, dogs, cars...)
    Took days to train on powerful GPUs

We are going to:
    KEEP all learned features (weights)
    REPLACE only the last layer
    TRAIN only the last layer on our leaf data

Result: Great accuracy with minimal training time! 


---
##  Cell 2: What is MobileNetV3?

MobileNetV3 is a **lightweight** CNN (Convolutional Neural Network) designed for:
- Mobile and edge devices (low memory, fast inference)
- High accuracy with fewer parameters
- Efficient computation

```
Comparison:
Model          Parameters    Accuracy
─────────────────────────────────────
ResNet50       25M params    76% top-1
VGG16          138M params   71% top-1
MobileNetV3    5.4M params   75% top-1  ← Small but powerful!
```

**Why MobileNetV3 for us?**
- Small → trains fast even on CPU
- Accurate enough for leaf disease detection
- Built into PyTorch's torchvision library

In [2]:
import torch
import torchvision.models as models

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")

print()
print(" Available MobileNet variants in torchvision:")
mobilenet_models = [m for m in dir(models) if 'mobilenet' in m.lower()]
for m in mobilenet_models:
    print(f"   - {m}")
print()
print("We will use: mobilenet_v3_small (fastest) or mobilenet_v3_large (more accurate)")

PyTorch version : 2.5.1+cu121
GPU available   : True
GPU name        : NVIDIA GeForce RTX 4050 Laptop GPU

 Available MobileNet variants in torchvision:
   - MobileNetV2
   - MobileNetV3
   - MobileNet_V2_Weights
   - MobileNet_V3_Large_Weights
   - MobileNet_V3_Small_Weights
   - mobilenet
   - mobilenet_v2
   - mobilenet_v3_large
   - mobilenet_v3_small
   - mobilenetv2
   - mobilenetv3

We will use: mobilenet_v3_small (fastest) or mobilenet_v3_large (more accurate)


---
## Cell 3: Load Pretrained MobileNetV3

PyTorch makes it incredibly easy to load a pretrained model.
One line of code downloads the model WITH weights trained on ImageNet!

In [3]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Load MobileNetV3 Small with pretrained ImageNet weights
# First time running this will download ~10MB of weights
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)

print("MobileNetV3 Small loaded with pretrained weights!")
print()
print("Model info:")

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"   Total parameters : {total_params:,}")
print(f"   In millions      : {total_params/1e6:.2f}M")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth
100%|██████████| 9.83M/9.83M [00:02<00:00, 4.48MB/s]

MobileNetV3 Small loaded with pretrained weights!

Model info:
   Total parameters : 2,542,856
   In millions      : 2.54M


---
##  Cell 4: Explore the Model Architecture

Let's look inside MobileNetV3 and understand its structure.
It has 3 main parts:
- `features` → extracts features from images (we FREEZE this)
- `avgpool`  → pools features into a single vector
- `classifier` → final classification layers (we REPLACE this)

In [4]:
# See the top level structure
print(" MobileNetV3 Top-Level Structure:")
print("=" * 50)
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"\n {name}")
    print(f"   Parameters : {params:,}")
    print(f"   Type       : {type(module).__name__}")

 MobileNetV3 Top-Level Structure:

 features
   Parameters : 927,008
   Type       : Sequential

 avgpool
   Parameters : 0
   Type       : AdaptiveAvgPool2d

 classifier
   Parameters : 1,615,848
   Type       : Sequential


In [5]:
# Look at the CLASSIFIER specifically
# This is the part we will REPLACE for our binary task
print(" Current Classifier (outputs 1000 classes for ImageNet):")
print("=" * 50)
print(model.classifier)
print()

# Find the input size of the last layer
last_layer = model.classifier[-1]
print(f"Last layer input  : {last_layer.in_features}")
print(f"Last layer output : {last_layer.out_features}  ← This is 1000 (ImageNet classes)")
print()
print("We need to change output from 1000 → 2 (Healthy or Diseased)")

 Current Classifier (outputs 1000 classes for ImageNet):
Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=1000, bias=True)
)

Last layer input  : 1024
Last layer output : 1000  ← This is 1000 (ImageNet classes)

We need to change output from 1000 → 2 (Healthy or Diseased)


---
##  Cell 5: What is Freezing Layers?

**Freezing** a layer means its weights will NOT be updated during training.

```
FROZEN layers   → weights stay the same (pretrained knowledge preserved)
UNFROZEN layers → weights get updated (learns our specific task)

Strategy:
  features    → FREEZE  (already knows how to detect edges, textures)
  classifier  → UNFREEZE (needs to learn healthy vs diseased)
```

This is called **Fine-tuning** - we fine-tune only the last layers.

In [6]:
# Check which parameters are trainable BEFORE freezing
print("BEFORE freezing:")
print("=" * 50)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")
print(f"Trainable %      : {trainable/total*100:.1f}%")
print()

# FREEZE all layers in features
for param in model.features.parameters():
    param.requires_grad = False   # ← This freezes the layer

# Check AFTER freezing
print("AFTER freezing features:")
print("=" * 50)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")
print(f"Trainable %      : {trainable/total*100:.1f}%")
print()
print("Only the classifier layers will be updated during training!")

BEFORE freezing:
Trainable params : 2,542,856
Total params     : 2,542,856
Trainable %      : 100.0%

AFTER freezing features:
Trainable params : 1,615,848
Total params     : 2,542,856
Trainable %      : 63.5%

Only the classifier layers will be updated during training!


---
##  Cell 6: Replace the Classifier

The original classifier outputs **1000 classes** (ImageNet).
We need to replace it to output **2 classes** (Healthy / Diseased).

In [8]:
import torch.nn as nn

# Get the input size of the classifier
in_features = model.classifier[-1].in_features
print(f"Classifier input features: {in_features}")
print()

# Replace the last linear layer
# FROM: Linear(in_features, 1000)  → 1000 ImageNet classes
# TO  : Linear(in_features, 2)     → 2 classes (Healthy/Diseased)
model.classifier[-1] = nn.Linear(in_features, 2)

print(" Classifier replaced!")
print()
print(" New Classifier:")
print(model.classifier)
print()
print(f"Now outputs 2 values:")
print(f"   Index 0 → Healthy score")
print(f"   Index 1 → Diseased score")

Classifier input features: 1024

 Classifier replaced!

 New Classifier:
Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=2, bias=True)
)

Now outputs 2 values:
   Index 0 → Healthy score
   Index 1 → Diseased score


---
##  Cell 7: Test a Forward Pass

A **forward pass** means feeding data through the model and getting output.
Let's feed one fake image and see what comes out!

In [9]:
# Create a fake image tensor (batch of 1 image)
# Shape: [batch=1, channels=3, height=224, width=224]
fake_image = torch.randn(1, 3, 224, 224)

# Set model to evaluation mode
model.eval()

# Forward pass — feed image through model
with torch.no_grad():   # no_grad = don't calculate gradients (saves memory)
    output = model(fake_image)

print(" Forward Pass Test:")
print("=" * 50)
print(f"Input shape  : {fake_image.shape}")
print(f"Output shape : {output.shape}   ← [batch=1, classes=2]")
print(f"Raw output   : {output}")
print()

# Convert raw output to probabilities using Softmax
probabilities = torch.softmax(output, dim=1)
print(f"Probabilities: {probabilities}")
print(f"   Healthy  : {probabilities[0][0].item()*100:.1f}%")
print(f"   Diseased : {probabilities[0][1].item()*100:.1f}%")
print()
print("(These are random since we haven't trained yet!)")

 Forward Pass Test:
Input shape  : torch.Size([1, 3, 224, 224])
Output shape : torch.Size([1, 2])   ← [batch=1, classes=2]
Raw output   : tensor([[-0.0678,  0.3223]])

Probabilities: tensor([[0.4037, 0.5963]])
   Healthy  : 40.4%
   Diseased : 59.6%

(These are random since we haven't trained yet!)


---
##  Cell 8: Summary - What We Have So Far

In [10]:
print(" Model Summary")
print("=" * 50)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Model          : MobileNetV3 Small")
print(f"Pretrained on  : ImageNet (1.2M images, 1000 classes)")
print(f"Our task       : Binary (Healthy vs Diseased)")
print()
print(f"Total params   : {total:,}")
print(f"Frozen params  : {frozen:,}  (features — not updated)")
print(f"Trainable      : {trainable:,}  (classifier - gets updated)")
print()
print("Strategy:")
print("   features   → FROZEN   (pretrained knowledge preserved)")
print("   classifier → TRAINABLE (learns our specific task)")
print()
print(" Step 3 Complete!")
print("  Next: Step 4 — Build & Save the Model properly")

 Model Summary
Model          : MobileNetV3 Small
Pretrained on  : ImageNet (1.2M images, 1000 classes)
Our task       : Binary (Healthy vs Diseased)

Total params   : 1,519,906
Frozen params  : 927,008  (features — not updated)
Trainable      : 592,898  (classifier - gets updated)

Strategy:
   features   → FROZEN   (pretrained knowledge preserved)
   classifier → TRAINABLE (learns our specific task)

 Step 3 Complete!
  Next: Step 4 — Build & Save the Model properly


---
##  Step 3 Summary - What You Learned

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| **Transfer Learning** | Reuse a model trained on another task | Saves time, needs less data |
| **MobileNetV3** | Lightweight CNN pretrained on ImageNet | Fast, accurate, built into PyTorch |
| **Pretrained weights** | Parameters already optimized | Already knows edges, textures, shapes |
| **Freezing layers** | Stop certain layers from updating | Preserve learned features |
| **Fine-tuning** | Train only the last layers | Adapt to our specific task |
| **Forward pass** | Feed image → get prediction | Core operation of any neural network |
| **Softmax** | Converts raw scores → probabilities | Makes output interpretable |

---
